# 🏗️ Proiect End-to-End: Analiza Pieței Imobiliare (Storia.ro)
**Autor:** Bahna Darius | **Arhitectură:** Medallion (Bronze -> Silver)

In [0]:
from pyspark.sql.types import IntegerType, DoubleType
import pyspark.sql.functions as F

## 🥉 Stratul Bronze (Ingestie)
**Obiectiv:** Citirea datelor brute din sursa externă, fără nicio modificare (Schema-on-Read).

In [0]:
path_data = "/Volumes/workspace/default/raw_data/storia_raw_data.json"
df_raw = spark.read.json(path_data, multiLine=True)

display(df_raw.limit(5))

area,listing_id,location,price,rooms,scraped_at,source,title,url
45.4 m²,3dba40f38d59625c5b9db7e27fef67a8,"Drumul Taberei, Sectorul 6, Bucuresti",115 000 €,2 camere,2026-05-23T07:14:18.655326+00:00,storia.ro,2 camere bloc 2025 - Novum Timișoara,https://www.storia.ro/hpr/ro/oferta/2-camere-bloc-2025-novum-timioara-IDHc0Y
55 m²,0b25b3f9cd94e1be04c84a5a5e7e832d,"Bulevardul Prof. Dimitrie Pompeiu, Pipera, Sectorul 2, Bucuresti",200 000 €,2 camere,2026-05-23T07:14:23.433418+00:00,storia.ro,2 camere Nusco City I Finalizat I Contract Vanzare I COMISION 0%,https://www.storia.ro/hpr/ro/oferta/2-camere-nusco-city-i-finalizat-i-contract-vanzare-i-comision-0-IDGkPj
112 m²,5e07a21f004355c94474bbe11c44797d,"Floreasca, Sectorul 1, Bucuresti",395 000 €,4 camere,2026-05-23T07:14:27.092030+00:00,storia.ro,4 Camere Floreasca | Et.2 | Mobilat Utilat |,https://www.storia.ro/hpr/ro/oferta/4-camere-floreasca-et-2-mobilat-utilat-IDHiBg
71 m²,e95f48797bcd6ec571f925a9427e8cca,"Strada Matei Basarab 65, Centrul Civic, Sectorul 3, Bucuresti",235 000 €,3 camere,2026-05-23T07:14:30.332310+00:00,storia.ro,"Apartament 3 camere, Matei Basarab, 0% Comision Cumparator",https://www.storia.ro/hpr/ro/oferta/apartament-3-camere-matei-basarab-0-comision-cumparator-IDHdZ1
56.01 m²,caff9b3520f47bab2c5bb35520027e7f,"Titan, Sectorul 3, Bucuresti",103 300 €,2 camere,2026-05-23T07:14:35.221124+00:00,storia.ro,2 camere–Auchan Titan-Mutare rapida-Acces rapid Transport,https://www.storia.ro/hpr/ro/oferta/2-camereauchan-titan-mutare-rapida-acces-rapid-transport-IDHcBG


## 🥈 Stratul Silver (Curățare, Tipizare și Îmbogățire)
**Obiectiv:** Standardizarea datelor, extragerea metricilor matematice și gestionarea anomaliilor (try_cast).

In [0]:
df_silver = df_raw.withColumn(
    "Price_EUR",
    F.expr("try_cast(regexp_replace(price, '[^0-9]', '') AS INT)")
).withColumn(
    "Area_sqm",
    F.expr("try_cast(regexp_extract(area, '([0-9.]+)', 1) AS DOUBLE)")
).withColumn(
    "Price_per_sqm",
    F.round(
        (F.col("Price_EUR") / F.col("Area_sqm")), 2
    )
)

display(df_silver.select("price", "Price_EUR", "area", "Area_sqm", "Price_per_sqm").limit(5))

price,Price_EUR,area,Area_sqm,Price_per_sqm
115 000 €,115000,45.4 m²,45.4,2533.04
200 000 €,200000,55 m²,55.0,3636.36
395 000 €,395000,112 m²,112.0,3526.79
235 000 €,235000,71 m²,71.0,3309.86
103 300 €,103300,56.01 m²,56.01,1844.31


In [0]:
df_silver = df_silver.withColumn(
    "rooms",
    F.expr("try_cast(regexp_extract(rooms, '([0-9]+)', 1) AS INT)")
)

display(df_silver.select("rooms").limit(5))

rooms
2
2
4
3
2


In [0]:
# Pasul 1: Tratăm Empty Strings ("") la locația brută
df_silver = df_silver.withColumn(
    "location",
    F.when(F.trim(F.col("location")) == "", F.lit(None)).otherwise(F.trim(F.col("location")))
)

# Pasul 2: Extragerea inteligentă a Sectorului (Regex)
df_silver = df_silver.withColumn(
    "City_Sector",
    F.regexp_extract(F.col("location"), r"(?i)(sector(?:ul)?\s*[1-6])", 1)
).withColumn(
    # FIX: Dacă regex-ul nu a găsit nimic (șir gol), îl transformăm explicit în null
    "City_Sector",
    F.when(F.col("City_Sector") == "", F.lit(None)).otherwise(F.col("City_Sector"))
)

# Pasul 3: Cartierele curate
df_silver = df_silver.withColumn(
    "Neighborhood",
    # Mai întâi ștergem Sectorul din text
    F.regexp_replace(F.col("location"), r"(?i),?\s*sector(?:ul)?\s*[1-6]", "")
).withColumn(
    "Neighborhood",
    # Apoi ștergem cuvintele București sau Ilfov pentru a lăsa cartierul complet curat
    # F.trim curăță virgulele sau spațiile lăsate în urmă.
    F.trim(F.regexp_replace(F.col("Neighborhood"), r"(?i),?\s*(Bucure[sșţt]ti|Ilfov)", ""))
)

# Afișăm rezultatul
display(df_silver.select("location", "Neighborhood", "City_Sector").limit(10))

location,Neighborhood,City_Sector
"Drumul Taberei, Sectorul 6, Bucuresti",Drumul Taberei,Sectorul 6
"Bulevardul Prof. Dimitrie Pompeiu, Pipera, Sectorul 2, Bucuresti","Bulevardul Prof. Dimitrie Pompeiu, Pipera",Sectorul 2
"Floreasca, Sectorul 1, Bucuresti",Floreasca,Sectorul 1
"Strada Matei Basarab 65, Centrul Civic, Sectorul 3, Bucuresti","Strada Matei Basarab 65, Centrul Civic",Sectorul 3
"Titan, Sectorul 3, Bucuresti",Titan,Sectorul 3
"IMGB, Sectorul 4, Bucuresti",IMGB,Sectorul 4
"Rahova, Sectorul 5, Bucuresti",Rahova,Sectorul 5
"Rahova, Sectorul 5, Bucuresti",Rahova,Sectorul 5
"Rahova, Sectorul 5, Bucuresti",Rahova,Sectorul 5
"Theodor Pallady, Sectorul 3, Bucuresti",Theodor Pallady,Sectorul 3


### 🛡️ Data Quality Checks (Validarea Calității Datelor)
**Obiectiv:** Verificarea consistenței datelor înainte de a le salva în Data Lake.

In [0]:
null_counts_data = df_silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_silver.columns
])

display(null_counts_data)

area,listing_id,location,price,rooms,scraped_at,source,title,url,Price_EUR,Area_sqm,Price_per_sqm,City_Sector,Neighborhood
0,0,0,0,2,0,0,0,0,0,0,0,101,0


### 🏛️ Decizie Arhitecturală: Păstrarea Valorilor Null (Adevărul Curățat)

**Regula Medallion:** În Stratul Silver **NU** ștergem rândurile care conțin valori lipsă (`null`) pe anumite coloane (ex: `City_Sector`, `rooms`). 

**Motivația de Business:** Dacă am șterge un rând doar pentru că îi lipsește sectorul, am pierde automat și prețul sau suprafața acelui apartament. Acest lucru ar distruge complet rapoartele globale din Stratul Gold (ex: "Valoarea Totală a Pieței"). 

*Tratarea valorilor lipsă (prin excludere sau mascare cu `COALESCE`) este responsabilitatea exclusivă a Stratului Gold, adaptată pentru fiecare raport SQL în parte.*

In [0]:
df_silver = df_silver.drop("price", "location", "area")

display(df_silver.limit(5))

listing_id,rooms,scraped_at,source,title,url,Price_EUR,Area_sqm,Price_per_sqm,City_Sector,Neighborhood
3dba40f38d59625c5b9db7e27fef67a8,2,2026-05-23T07:14:18.655326+00:00,storia.ro,2 camere bloc 2025 - Novum Timișoara,https://www.storia.ro/hpr/ro/oferta/2-camere-bloc-2025-novum-timioara-IDHc0Y,115000,45.4,2533.04,Sectorul 6,Drumul Taberei
0b25b3f9cd94e1be04c84a5a5e7e832d,2,2026-05-23T07:14:23.433418+00:00,storia.ro,2 camere Nusco City I Finalizat I Contract Vanzare I COMISION 0%,https://www.storia.ro/hpr/ro/oferta/2-camere-nusco-city-i-finalizat-i-contract-vanzare-i-comision-0-IDGkPj,200000,55.0,3636.36,Sectorul 2,"Bulevardul Prof. Dimitrie Pompeiu, Pipera"
5e07a21f004355c94474bbe11c44797d,4,2026-05-23T07:14:27.092030+00:00,storia.ro,4 Camere Floreasca | Et.2 | Mobilat Utilat |,https://www.storia.ro/hpr/ro/oferta/4-camere-floreasca-et-2-mobilat-utilat-IDHiBg,395000,112.0,3526.79,Sectorul 1,Floreasca
e95f48797bcd6ec571f925a9427e8cca,3,2026-05-23T07:14:30.332310+00:00,storia.ro,"Apartament 3 camere, Matei Basarab, 0% Comision Cumparator",https://www.storia.ro/hpr/ro/oferta/apartament-3-camere-matei-basarab-0-comision-cumparator-IDHdZ1,235000,71.0,3309.86,Sectorul 3,"Strada Matei Basarab 65, Centrul Civic"
caff9b3520f47bab2c5bb35520027e7f,2,2026-05-23T07:14:35.221124+00:00,storia.ro,2 camere–Auchan Titan-Mutare rapida-Acces rapid Transport,https://www.storia.ro/hpr/ro/oferta/2-camereauchan-titan-mutare-rapida-acces-rapid-transport-IDHcBG,103300,56.01,1844.31,Sectorul 3,Titan


## 🥈 Stratul Silver (Salvarea DAtelor)
**Obiectiv:** SAlvare a stratului Silver

In [0]:
df_silver.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/raw_data/silver_storia")

display(df_silver.limit(5))

listing_id,rooms,scraped_at,source,title,url,Price_EUR,Area_sqm,Price_per_sqm,City_Sector,Neighborhood
3dba40f38d59625c5b9db7e27fef67a8,2,2026-05-23T07:14:18.655326+00:00,storia.ro,2 camere bloc 2025 - Novum Timișoara,https://www.storia.ro/hpr/ro/oferta/2-camere-bloc-2025-novum-timioara-IDHc0Y,115000,45.4,2533.04,Sectorul 6,Drumul Taberei
0b25b3f9cd94e1be04c84a5a5e7e832d,2,2026-05-23T07:14:23.433418+00:00,storia.ro,2 camere Nusco City I Finalizat I Contract Vanzare I COMISION 0%,https://www.storia.ro/hpr/ro/oferta/2-camere-nusco-city-i-finalizat-i-contract-vanzare-i-comision-0-IDGkPj,200000,55.0,3636.36,Sectorul 2,"Bulevardul Prof. Dimitrie Pompeiu, Pipera"
5e07a21f004355c94474bbe11c44797d,4,2026-05-23T07:14:27.092030+00:00,storia.ro,4 Camere Floreasca | Et.2 | Mobilat Utilat |,https://www.storia.ro/hpr/ro/oferta/4-camere-floreasca-et-2-mobilat-utilat-IDHiBg,395000,112.0,3526.79,Sectorul 1,Floreasca
e95f48797bcd6ec571f925a9427e8cca,3,2026-05-23T07:14:30.332310+00:00,storia.ro,"Apartament 3 camere, Matei Basarab, 0% Comision Cumparator",https://www.storia.ro/hpr/ro/oferta/apartament-3-camere-matei-basarab-0-comision-cumparator-IDHdZ1,235000,71.0,3309.86,Sectorul 3,"Strada Matei Basarab 65, Centrul Civic"
caff9b3520f47bab2c5bb35520027e7f,2,2026-05-23T07:14:35.221124+00:00,storia.ro,2 camere–Auchan Titan-Mutare rapida-Acces rapid Transport,https://www.storia.ro/hpr/ro/oferta/2-camereauchan-titan-mutare-rapida-acces-rapid-transport-IDHcBG,103300,56.01,1844.31,Sectorul 3,Titan
